# 🛡️ Spam SMS / Email Classifier
### Complete Machine Learning Pipeline: EDA, TF-IDF N-grams, Naive Bayes Baseline, Model Benchmarking, and Explainable Inference

**Author:** Data Science & NLP Engineer  
**Dataset:** UCI SMS Spam Collection Corpus  
**Task:** Binary Text Classification (`ham` $\rightarrow$ 0, `spam` $\rightarrow$ 1)  

---
## 1. Project Objective & Methodology
The objective of this project is to develop an end-to-end, reproducible Natural Language Processing (NLP) classifier that accurately identifies whether an incoming SMS or email message is **Spam** (unsolicited commercial/fraudulent message) or **Ham** (legitimate personal/transactional message).

### Key Priorities:
1. **High Precision on Spam**: In communication filtering, false positives (marking legitimate messages as spam) can lead to missed critical emails or urgent personal SMS. We strive for $>98\%$ precision.
2. **Zero Data Leakage**: TF-IDF vocabulary and IDF weights are fit strictly on the training set, and exact duplicate text messages are purged to prevent memorization across splits.
3. **Calibrated Confidence Scores**: Return probabilistic confidence scores alongside binary labels to allow custom decision thresholding in production.

In [ ]:
# Environment setup and imports
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score
)

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
RANDOM_STATE = 42
print('Environment and libraries loaded successfully.')

## 2. Dataset Loading and Exploratory Data Analysis (EDA)
We use the well-established **UCI SMS Spam Collection** corpus, which consists of 5,574 English SMS messages compiled from multiple sources including Grumbletext (UK spam complaints forum), Caroline Tagg's PhD thesis, and the NUS SMS Corpus.

In [ ]:
data_path = os.path.join('..', 'data', 'SMSSpamCollection')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'SMSSpamCollection')

# Load tab-separated file
df = pd.read_csv(data_path, sep='\t', names=['raw_label', 'text'], quoting=3, encoding='utf-8')

print(f'Raw Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
print('\nMissing values:\n', df.isnull().sum())
print('\nRaw Label Distribution:\n', df['raw_label'].value_counts())
df.head()

### Duplicate Inspection & Data Leakage Prevention
Spam campaigns frequently reuse identical copy (e.g. "*Claim your prize*", "*Free entry in 2 a wkly comp*"). Similarly, short conversational messages like "*Ok*" appear repeatedly.

If identical rows are randomly partitioned into both training and testing sets, the test set evaluation will measure memorization rather than generalization. **Therefore, we remove exact duplicate messages.**

In [ ]:
duplicates_count = df.duplicated(subset=['text']).sum()
print(f'Exact duplicate messages identified: {duplicates_count}')

# Map labels consistently: spam -> 1, ham -> 0
df['label'] = df['raw_label'].str.strip().str.lower().map({'spam': 1, 'ham': 0})

# Deduplicate text column
df_clean = df.drop_duplicates(subset=['text']).copy().reset_index(drop=True)
print(f'Clean unique corpus size: {len(df_clean)} messages')
print('\nClean Class Distribution:')
class_dist = df_clean['label'].value_counts()
print(f'  Ham (0):  {class_dist[0]} ({class_dist[0]/len(df_clean):.1%})')
print(f'  Spam (1): {class_dist[1]} ({class_dist[1]/len(df_clean):.1%})')

In [ ]:
# Feature Engineering for Visual EDA
df_clean['char_length'] = df_clean['text'].apply(len)
df_clean['word_count'] = df_clean['text'].apply(lambda x: len(x.split()))
df_clean['has_currency'] = df_clean['text'].str.contains(r'[$£€]', regex=True).astype(int)
df_clean['has_digits'] = df_clean['text'].str.contains(r'\d', regex=True).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Length distribution
sns.histplot(data=df_clean, x='char_length', hue='raw_label', bins=60, kde=True, ax=axes[0], palette={'ham': '#2563EB', 'spam': '#DC2626'})
axes[0].set_title('Message Character Length Distribution by Class', fontsize=12, fontweight='bold')
axes[0].set_xlim(0, 300)

# Currency symbol frequency
currency_rates = df_clean.groupby('raw_label')['has_currency'].mean() * 100
sns.barplot(x=currency_rates.index, y=currency_rates.values, ax=axes[1], palette={'ham': '#2563EB', 'spam': '#DC2626'})
axes[1].set_title('% of Messages Containing Currency Symbols ($/£/€)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')

plt.tight_layout()
plt.show()

**Key EDA Insights:**
1. **Message Length**: Spam messages are significantly longer on average (~140 characters, approaching the 160-char SMS limit) compared to legitimate ham messages (~70 characters).
2. **Specific Spam Signals**: Currency symbols ($/£/€), urgent call-to-action words, and phone numbers occur at vastly higher rates in spam.

## 3. Stratified Train / Test Split
We partition the dataset into **80% Training** (4,136 messages) and **20% Testing** (1,035 messages) using stratified sampling to preserve the 87.4% / 12.6% class ratio.

In [ ]:
X_raw = np.array(df_clean['text'].tolist(), dtype=object)
y_raw = np.array(df_clean['label'].tolist(), dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y_raw,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_raw
)

print(f'Training Set: {len(X_train)} samples (Ham: {(y_train == 0).sum()}, Spam: {(y_train == 1).sum()})')
print(f'Testing Set:  {len(X_test)} samples (Ham: {(y_test == 0).sum()}, Spam: {(y_test == 1).sum()})')

## 4. Pipeline Construction & Model Training
We encapsulate feature extraction (`TfidfVectorizer`) and classification in a `scikit-learn Pipeline`.
- **TF-IDF Configuration**: Includes unigrams and bigrams (`ngram_range=(1, 2)`), preserves currency symbols and punctuation flags via custom token pattern, with `min_df=2`.
- **Models Evaluated**:
  1. **Multinomial Naive Bayes (Baseline)**: Classic, fast, highly effective for bag-of-words text classification.
  2. **Logistic Regression (Extension)**: Linear model with balanced class weighting.
  3. **Complement Naive Bayes (Extension)**: Adaptation of Multinomial NB tailored for imbalanced text classes.
  4. **Linear Support Vector Classifier (Extension)**: High-margin linear classifier with probability calibration.

In [ ]:
token_pattern = r'(?u)\b\w+\b|[!?$£€]'

models = {
    'Multinomial Naive Bayes (Baseline)': Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, token_pattern=token_pattern)),
        ('clf', MultinomialNB(alpha=0.1))
    ]),
    'Logistic Regression (Extension)': Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, token_pattern=token_pattern)),
        ('clf', LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
    ]),
    'Complement Naive Bayes': Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, token_pattern=token_pattern)),
        ('clf', ComplementNB(alpha=0.2))
    ]),
    'Calibrated Linear SVC': Pipeline([
        ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2, token_pattern=token_pattern)),
        ('clf', CalibratedClassifierCV(LinearSVC(random_state=RANDOM_STATE, dual='auto'), cv=3))
    ])
}

results = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (Spam)': precision_score(y_test, y_pred, pos_label=1),
        'Recall (Spam)': recall_score(y_test, y_pred, pos_label=1),
        'F1-Score (Spam)': f1_score(y_test, y_pred, pos_label=1),
        'ROC-AUC': auc(*roc_curve(y_test, y_proba)[:2]),
        'y_pred': y_pred,
        'y_proba': y_proba
    }

# Display benchmark comparison table
benchmark_df = pd.DataFrame([
    {'Model': k, **{m: f'{v:.2%}' for m, v in vals.items() if m not in ['y_pred', 'y_proba']}}
    for k, vals in results.items()
])
benchmark_df

## 5. Evaluation & Confusion Matrix Analysis
Let us examine the performance of our primary baseline model (**Multinomial Naive Bayes**).

In [ ]:
primary_model = models['Multinomial Naive Bayes (Baseline)']
y_pred_nb = results['Multinomial Naive Bayes (Baseline)']['y_pred']
cm = confusion_matrix(y_test, y_pred_nb)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Predicted Ham (0)', 'Predicted Spam (1)'],
    yticklabels=['Actual Ham (0)', 'Actual Spam (1)'],
    cbar=False,
    annot_kws={'size': 14}
)
plt.title('Confusion Matrix: Multinomial Naive Bayes on Test Set (N=1,035)', fontsize=12, pad=12)
plt.tight_layout()
plt.show()

print('Detailed Classification Report:')
print(classification_report(y_test, y_pred_nb, target_names=['Ham', 'Spam'], digits=4))

### ROC and Precision-Recall Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    roc_score = auc(fpr, tpr)
    ax1.plot(fpr, tpr, lw=2, label=f"{name.split(' (')[0]} (AUC={roc_score:.3f})")
    
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    pr_score = average_precision_score(y_test, res['y_proba'])
    ax2.plot(rec, prec, lw=2, label=f"{name.split(' (')[0]} (AP={pr_score:.3f})")

ax1.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6)
ax1.set_title('Receiver Operating Characteristic (ROC)', fontsize=12, fontweight='bold')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate (Recall)')
ax1.legend(loc='lower right')

ax2.set_title('Precision-Recall Curve (Spam = Positive)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Recall (Spam Caught)')
ax2.set_ylabel('Precision (Spam Accuracy)')
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

## 6. Error Analysis: Inspecting Misclassifications
Analyzing failure modes provides critical insights into model behavior:
- **False Positives (FP)**: Legitimate Ham messages misflagged as Spam (2 messages).
- **False Negatives (FN)**: Spam messages that slipped into inbox (11 messages).

In [ ]:
# Inspect False Positives
fp_indices = np.where((y_test == 0) & (y_pred_nb == 1))[0]
print(f'=== FALSE POSITIVES ({len(fp_indices)} messages) ===')
for i, idx in enumerate(fp_indices, 1):
    prob = primary_model.predict_proba([X_test[idx]])[0][1]
    print(f"{i}. Message: \"{X_test[idx]}\"")
    print(f"   Spam Probability: {prob:.2%}\n")

# Inspect Sample False Negatives
fn_indices = np.where((y_test == 1) & (y_pred_nb == 0))[0]
print(f'=== SAMPLE FALSE NEGATIVES ({len(fn_indices)} total, showing first 5) ===')
for i, idx in enumerate(fn_indices[:5], 1):
    prob = primary_model.predict_proba([X_test[idx]])[0][1]
    print(f"{i}. Message: \"{X_test[idx]}\"")
    print(f"   Spam Probability: {prob:.2%}\n")

## 7. Model Predictions on 10 Benchmark Test Cases
We now evaluate the model on the required 10 illustrative test cases from the project roadmap, plus 3 real-world edge cases.

In [ ]:
test_cases = [
    'Congratulations! You have won a prize. Claim now.',
    'Are we still meeting at 5 pm?',
    'URGENT: claim your free reward today!',
    'Please send me the notes from class.',
    'Your account has been selected for a bonus.',
    'I will call you when I reach home.',
    'Limited offer! Click the link to get a discount.',
    'Can you share the assignment file?',
    'You have been selected for a cash reward.',
    'Thanks, see you tomorrow.',
    'Hey, can you review the pull request on GitHub before tonight?',
    'IMPORTANT: Your parcel delivery failed. Update your card details at http://scam-delivery.link',
    'Your Google verification code is 482910. Do not share this with anyone.'
]

prediction_rows = []
classes = list(primary_model.classes_)
for idx, msg in enumerate(test_cases, 1):
    pred_label = primary_model.predict([msg])[0]
    probas = primary_model.predict_proba([msg])[0]
    confidence = probas[classes.index(pred_label)]
    prediction_rows.append({
        '#': idx,
        'Input Message': msg,
        'Predicted Label': 'Spam' if pred_label == 1 else 'Ham',
        'Confidence': f'{confidence:.2%}',
        'Spam Probability': f'{probas[1]:.2%}'
    })

pred_df = pd.DataFrame(prediction_rows)
pred_df

## 8. Model Serialization & Round-Trip Verification
We serialize the complete pipeline using `joblib` into `models/spam_classifier.joblib`.

In [ ]:
export_path = os.path.join('..', 'models', 'spam_classifier.joblib')
if not os.path.exists(os.path.dirname(export_path)):
    export_path = os.path.join('models', 'spam_classifier.joblib')

# Save model pipeline
joblib.dump(primary_model, export_path)
print(f'Model pipeline saved to: {export_path}')

# Reload and verify round-trip inference
loaded_model = joblib.load(export_path)
test_msg = ['URGENT: Claim your free £1000 prize now!']
lbl = loaded_model.predict(test_msg)[0]
conf = loaded_model.predict_proba(test_msg)[0][lbl]
print(f'Verification Test: Label = {"Spam" if lbl == 1 else "Ham"}, Confidence = {conf:.2%}')